# **Mapping Knowledge Flow and Bottlenecks in IT Service Desk Ticket Resolution**


## Executive Summary

Standard IT Service Management analytics often focus on productivity metrics such as ticket volume, mean time to resolution, re-open rates and a whole library of flat, one-dimensional measures (How much? How often? How long?). While these provide a snapshot of throughput, they fail to show the complex structural interactions of how knowledge is distributed and how work flows through the organization. This project combines Natural Language Processing and Social Network Analysis to model the service desk as a dynamic interaction network. This analysis seeks to identify the informal organization uncovering hidden expertise, structural dependencies, and operational risks that are invisible in standard linear reporting at a NYC college.


Research Question & Hypothesis

- Guiding Question: How does resolution knowledge flow across technicians, end users, and issue categories, and can network graph metrics identify critical knowledge brokers or information bottlenecks.

- Hypothesis: IT support knowledge is non-uniformly distributed; a minority of technicians serve as informal knowledge brokers or structural bottlenecks, possessing high betweenness centrality for specific, high-impact technical domains. Likewise, a select number of repeat or influential end users and issue categories may have outsized influence over the information flow.


## Environment Setup and Data Ingestion

We establish our environment by loading libraries and importing the anonymized IT service desk dataset.  The output displays critical metadata including ticket categories, assignment groups, and arrays of contributing technicians for specific hardware and software incidents.

## A Note About Anonymization of the Dataset

The dataset itself is not available for public access on github.  The dataset is an internal ITSM database from an NYC college which contains personally identifiable information (PII) within the work notes, comments and short descriptions columns. These columns were reserved for the NLP analysis.  The other columns which contained names were anonymized.  For the staff technicians, I use Disney character names so they are easy to detect in the data and visualizations (eg, Mickey Mouse). For the student technicians and other employee references, these names were generically lableled and numbered (eg, student_technician1, student_technician2, etc)

In [1]:
# Import modules
import os
import warnings
import pandas as pd
from google.colab import drive
warnings.filterwarnings("ignore", message="Could not infer format")

# Mount Google Drive environment
drive.mount('/content/drive', force_remount=True)

# Set the working directory
os.chdir("/content/drive/MyDrive/Data620")

# Load the anonymized master dataset
df = pd.read_csv("final_tixdata_master_anonymized.csv")

# Define the text columns for NLP processing
nlp_columns = ['work_notes', 'comments', 'short_description']

# Display the dataset
display(df.drop(columns=nlp_columns).head(10))

Mounted at /content/drive


,number,sys_updated_on,assigned_to,sys_updated_by,contact_type,u_on_behalf_of,sys_created_on,assignment_group,closed_at,u_subcategory,u_symptom,state,opened_at,opened_by,sys_mod_count,reopen_count,reassignment_count,is_open,contributing_technicians,technician_count
0,INC0112581,11/2/22 13:04,Jiminy Cricket,system8,Email,staff_requester176,11/12/21 10:49,Service Desk,NaN,Computer (Desktop/Laptop),Install/New,Ready for Release,11/12/21 10:43,Aladdin Hassim,24,2,0,True,Aladdin Hassim|Jiminy Cricket|Mulan Fa|Andy Davis,4
1,INC0113425,10/28/22 15:36,Sid Phillips,Sid Phillips,Email,Chesire Cat,12/22/21 10:07,Enterprise Applications,NaN,Other/Not Listed,Configure/Modify,Work in Progress,12/22/21 10:06,Chesire Cat,6,0,0,True,Chesire Cat|Tiana Rogers|Sid Phillips,3
2,INC0110699,8/10/22 22:00,Jiminy Cricket,system8,Phone,Jiminy Cricket,9/13/21 10:24,Service Desk,8/10/22 22:00,Account/Identity Mgmt,Password,Closed Skipped,9/13/21 9:44,Mulan Fa,26,0,0,False,Minnie Mouse|Jiminy Cricket|Mulan Fa,3
3,INC0112100,10/21/22 12:00,Mulan Fa,system8,Email,staff_requester1482,10/25/21 12:35,Service Desk,10/21/22 12:00,Monitor,Install/New,Closed Skipped,10/25/21 12:33,Mulan Fa,5,0,1,False,Minnie Mouse|Jiminy Cricket|Mulan Fa,3
4,INC0112393,10/20/22 11:00,Jiminy Cricket,system8,Email,staff_requester167,11/5/21 15:26,Service Desk,10/20/22 11:00,Computer (Desktop/Laptop),Install/New,Closed Skipped,11/5/21 15:26,staff_requester994,101,1,10,False,Huey Duck|Minnie Mouse|Jiminy Cricket|Tinker B...,6
5,INC0113478,2/11/22 17:00,Aladdin Hassim,system8,Email,staff_requester1028,12/31/21 13:26,Service Desk,2/11/22 17:00,Account/Identity Mgmt,Error/Bug,Closed Complete,12/31/21 13:26,staff_requester1028,9,0,0,False,Aladdin Hassim|Andy Davis,2
6,INC0113474,1/22/22 10:00,Mary Poppins,system8,Email,student_requester722,12/30/21 8:43,Student Computing,1/22/22 10:00,Account/Identity Mgmt,Connectivity/Down,Closed Complete,12/30/21 8:43,student_requester722,8,0,0,False,Mary Poppins|Eve Probe,2
7,INC0113356,9/11/22 0:00,Andy Davis,system8,Email,staff_requester1324,12/17/21 10:11,Service Desk,9/11/22 0:00,Computer (Desktop/Laptop),Install/New,Closed Skipped,12/17/21 10:11,staff_requester796,34,0,4,False,Minnie Mouse|Aladdin Hassim|Mulan Fa|Andy Davi...,5
8,INC0113018,8/20/22 11:00,Huey Duck,system8,Walk-in,staff_requester372,12/2/21 13:13,Service Desk,8/20/22 11:00,Desktop Application,Configure/Modify,Closed Skipped,12/2/21 13:04,Tinker Bell,21,0,0,False,Huey Duck|Tinker Bell|Andy Davis|Mad Hatter,4
9,INC0113463,1/16/22 15:00,system8,system8,Email,student_requester1222,12/28/21 15:26,Student Computing,1/16/22 15:00,Account/Identity Mgmt,Configure/Modify,Closed Complete,12/28/21 15:26,student_requester1222,4,0,0,False,Eve Probe,1


## Natural Language Processing Pipeline

We construct the  natural language processing pipeline to transform the unstructured ticket communications into a refined list of core technical concepts. By combining descriptive text fields and applying NLP models, we extract foundational terms like nouns and adjectives while systematically stripping away URLs and standard stop-words.

The custom dictionary of domain noise filters out institutional jargon, email signatures, and dynamically extracted user names. This dictionary highlights the highly iterative nature of text mining;  it was  compiled only after **multiple** preliminary runs revealed localized conversational artifacts that were polluting the results. This pruning was essential for reducing the dimensionality of our dataset and ensures that our subsequent network models are built strictly upon actual IT infrastructure elements rather than the noise of daily communication.

Specifically, we applied these techniques to filter the language and create the pipeline:

- Entity Extraction: Isolates personnel names to protect anonymity and prevent human variables from skewing technical network nodes.

- Lexical Pruning: Removes institutional jargon, URLs, and conversational pleasantries to isolate the substantive technical signal.

- Part-of-Speech Tagging: Identifies and extracts core nouns and adjectives that represent the physical and software infrastructure.

- Dimensionality Reduction: Condenses a massive, unstructured vocabulary space into a viable dataset for topological modeling.


In [2]:
# Import libraries
import multiprocessing
import spacy
import pandas as pd
import re

# Load User Mapping Data
users_df = pd.read_csv('users_mapping.csv')
name_tokens = set()

# Split full names into individual lowercase tokens
for name in users_df['full_name'].dropna().str.lower():
    name_tokens.update(name.split())

# Split anonymized names and strip numerical identifiers
for name in users_df['anonymized_name'].dropna().str.lower():
    clean_name = ''.join([i for i in name if not i.isdigit()])
    name_tokens.update(clean_name.split('_'))

# Define standard domain noise institutional identifiers
domain_noise = {
    'pm', 'am', 'edt', 'est', 'ref', 'msg', 'li', 'br', 'ny', 'mon', 'fri',
    'please', 'thanks', 'thank', 'hello', 'hi', 'dear', 'regards',
    'best', 'ticket', 'issue', 'help', 'attached', 'forwarded',
    'user', 'customer', 'assigned', 'resolved', 'error', 'good',
    'morning', 'afternoon', 'question', 'concern', 'patience',
    'welcome', 'great', 'date', 'subject',
    'intended', 'recipient', 'dissemination', 'confidential',
    'privileged', 'confidentiality', 'privilege', 'exempt',
    'exemption', 'disclosure', 'law', 'material',
    'college', 'university', 'it', 'broadway', 'edu', 'new', 'york', 'inc',
    'null', 'png', 'image', 'laurels', 'milbank', 'milstein', 'gcollege', 'mycollege',
    'associate', 'director', 'chief', 'president', 'vice',
    'college', 'university', 'department', 'enterprise',
    'jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec',
    'tue', 'wed', 'thu', 'sat', 'sun',
    'urgent', 'matter', 'disclaimer', 'message', 'instruction', 'applicable'
}

# Merge static domain noise with  extracted user names
comprehensive_noise = domain_noise.union(name_tokens)

# Load core English language model
nlp = spacy.load("en_core_web_sm")

# Inject noise terms into the spaCy vocabulary
for word in comprehensive_noise:
    nlp.vocab[word].is_stop = True

# Combine the targeted NLP columns into a single comprehensive text feature
df['combined_text'] = df['short_description'].fillna('') + ' ' + \
                      df['work_notes'].fillna('') + ' ' + \
                      df['comments'].fillna('')

# Remove URLs and HTML
df['text_clean'] = df['combined_text'].str.replace(r'http\S+|www\.\S+', ' ', regex=True)
df['text_clean'] = df['text_clean'].str.replace(r'<.*?>', ' ', regex=True)
df['text_clean'] = df['text_clean'].str.replace(r'[^a-zA-Z\s]', ' ', regex=True)
df['text_clean'] = df['text_clean'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Define extraction logic
def extract_technical_lemmas(doc):
    valid_pos = {'NOUN', 'PROPN', 'ADJ'}
    invalid_ents = {'PERSON', 'ORG', 'GPE', 'LOC'}
    invalid_indices = {token.i for ent in doc.ents if ent.label_ in invalid_ents for token in ent}
    terms = []
    for token in doc:
        if token.i in invalid_indices:
            continue
        token_lower = token.text.lower()
        if token_lower in comprehensive_noise or token.is_stop or len(token_lower.strip()) <= 2:
            continue
        if token.pos_ in valid_pos:
            terms.append(token.lemma_.lower())
    return terms

# Execute optimized pipeline
num_cores = multiprocessing.cpu_count()
docs = nlp.pipe(df['text_clean'], disable=["parser"], batch_size=500, n_process=num_cores)

# Assign finalized lemmas to clean term data frame
df['cleaned_terms'] = [extract_technical_lemmas(doc) for doc in docs]
print("Text processing and term extraction complete.")

Text processing and term extraction complete.


## Statistical Thresholding and Technical Concept Extraction

To enhance our analysis from individual words to meaningful structural entities, we evaluate how frequently terms appear and co-occur across the service desk corpus. We first apply a Document Frequency (DF) threshold, which was critical for making significant gains in noise reduction by eliminating over **63%** of the vector space.  This was a critical step as prior to the DF threshold, domain noise was high and difficult to isolate and eliminate using the stop word lists.

We calculate the statistical likelihood of adjacent words forming paired concepts to combine terms like "adobe" and "creative" or "password" and "reset" into singular operational targets. Similar to our prior domain noise filtration, compiling the list of administrative bigrams to exclude was a highly iterative process, requiring multiple evaluation runs to strip out recurring procedural phrasing. The final output provides a clean list of core technical components.

Specifically, we applied these techniques to to our NLP pipeline:

- Document Frequency Thresholding: Establishes a baseline occurrence rate to drastically reduce dataset dimensionality and isolate structurally significant terms.

- Log-Likelihood Bigram Scoring: Identifies statistically significant word pairings to capture true technical concepts rather than coincidental phrasing.

- Iterative Administrative Filtration: Removes prevalent procedural combinations (e.g., "work_note", "service_desk") discovered through repeated testing.

- Multi-Word Tokenization: Connects pairs together into single network nodes, finalizing the bipartite targets required for topological modeling.

The resulting matrix displays the most statistically significant word pairings within the corpus, successfully capturing distinct IT infrastructure components such as "adobe creative," "password reset," and "docking station." The high log-likelihood scores validate that these pairings represent genuine operational concepts rather than coincidental conversational phrasing. By tokenizing these validated pairs into singular terms, we establish the definitive target nodes for our network architecture.

In [3]:
# Import libraries
from collections import Counter
import pandas as pd
import nltk
from nltk.collocations import BigramCollocationFinder
from nltk.metrics import BigramAssocMeasures
from nltk.tokenize import MWETokenizer

# Calculate Document Frequency
document_frequencies = Counter()
for term_list in df['cleaned_terms'].dropna():
    document_frequencies.update(set(term_list))

# Establish statistical floor to eliminate one-off typos and localized anomalies
min_df_threshold = 3

# Construct a verified vocabulary
verified_vocabulary = {term for term, count in document_frequencies.items() if count >= min_df_threshold}

# Calculate the noise reduction metrics
original_vocab_size = len(document_frequencies)
verified_vocab_size = len(verified_vocabulary)
reduction_percentage = ((original_vocab_size - verified_vocab_size) / original_vocab_size) * 100

# Output reduction metrics
print(f"Original Unique Terms: {original_vocab_size:,}")
print(f"Verified Structural Terms (DF >= {min_df_threshold}): {verified_vocab_size:,}")
print(f"Dimensionality Reduction: {reduction_percentage:.2f}% of the vector space eliminated.\n")

# Apply filtering to overwrite clean terms with verified terms
df['cleaned_terms'] = df['cleaned_terms'].apply(
    lambda x: [term for term in x if term in verified_vocabulary] if isinstance(x, list) else [])
corpus_terms = [term for term_list in df['cleaned_terms'] for term in term_list]

# Initialize bigram collocation finder
finder = BigramCollocationFinder.from_words(corpus_terms)

# Apply frequency threshold to remove  insignificant pairings
finder.apply_freq_filter(5)

# Define structural administrative bigrams to exclude
admin_bigrams = {
    ('additional', 'comment'), ('work', 'note'), ('service', 'desk'),
    ('comment', 'reply'), ('short', 'description'), ('comment', 'inactivity'),
    ('subject', 'incident'), ('department', 'code'), ('work', 'caller'),
    ('inactivity', 'caller'), ('business', 'day'), ('monday', 'friday'), ('long', 'usual'),
    ('time', 'slot'), ('likely', 'case'), ('place', 'correct'),
    ('incident', 'number'), ('phone', 'number'), ('email', 'address'),
    ('shipping', 'address'), ('caller', 'comment'), ('priority', 'moderate'),
    ('order', 'vendor'), ('current', 'situation'), ('previous', 'work'),
    ('opening', 'classification'), ('ready', 'deployment'), ('standard', 'ready'), ('machine', 'specialty'),
    ('machine', 'standard'), ('transfer', 'possible'), ('situation', 'long'), ('vendor', 'current'), ('usual', 'delivery'),
    ('delivery', 'day'), ('local', 'datum'), ('list', 'pre'), ('pre', 'approved'),
    ('specialty', 'installation'), ('machine', 'setup'), ('user', 'machine'),
    ('question', 'free'), ('feel', 'free'), ('urgent', 'matter'),
    ('disclaimer', 'message'), ('time', 'cause'), ('message', 'attached'),
    ('current', 'status'), ('subject', 'applicable'),
    ('material', 'use'), ('instruction', 'read'),
    ('correct', 'password'), ('lockout', 'minute'), ('attempt', 'minute'),
    ('accounts', 'login'), ('login', 'attempt'), ('minute', 'pass'),
    ('case', 'campus'), ('laptop', 'place'), ('cause', 'password'),
    ('tool', 'temporary'), ('length', 'helpdesk'),
    ('start', 'button'), ('right', 'corner'), ('dialog', 'box'),
    ('apple', 'menu'), ('finder', 'application'), ('drop', 'menu'),
    ('narrow', 'edge'),
    ('number', 'time'), ('moderate', 'current'), ('office', 'hour'),
    ('previous', 'worknote'), ('question', 'concern')
}

# Filter out specific administrative tuples
finder.apply_ngram_filter(lambda w1, w2: (w1, w2) in admin_bigrams)

# Instantiate association measures
bigram_measures = BigramAssocMeasures()

# Calculate log-likelihood ratios for remaining technical bigrams
top_bigrams = finder.score_ngrams(bigram_measures.likelihood_ratio)

# Construct dataframe to inspect extracted collocations
bigram_df = pd.DataFrame(top_bigrams, columns=['Bigram', 'Log_Likelihood_Score'])

# Extract the top 100 most statistically significant bigrams
target_bigrams = [tuple(x) for x in bigram_df['Bigram'].head(100)]

# Initialize the tokenizer to connect adjacent tokens together
mwe_tokenizer = MWETokenizer(target_bigrams, separator='_')

# Apply the tokenizer to the local records
df['network_nodes'] = df['cleaned_terms'].apply(lambda x: mwe_tokenizer.tokenize(x))

# Filter out remaining unigrams to focus strictly on complex nodes
def isolate_bigrams(token_list):
    return [token for token in token_list if '_' in token]

# Assign final targets to bipartite dataframe
df['bipartite_targets'] = df['network_nodes'].apply(isolate_bigrams)

# Display relevant bigrams
display(bigram_df.head(50))

Original Unique Terms: 21,013
Verified Structural Terms (DF >= 3): 7,746
Dimensionality Reduction: 63.14% of the vector space eliminated.



,Bigram,Log_Likelihood_Score
0,"(adobe, creative)",18368.211229
1,"(creative, cloud)",17587.882904
2,"(serial, number)",9663.606322
3,"(self, service)",8527.213372
4,"(password, reset)",8150.714553
5,"(pulse, secure)",7399.708333
6,"(support, session)",6689.190209
7,"(password, password)",6391.906692
8,"(data, transfer)",6132.580127
9,"(dbm, dbm)",5892.564890


## Bipartite Network Generation and Edge Weight Calculation

We construct the bipartite network architecture by linking technicians directly to the technical concepts they resolve. The dataset consist of every distinct interaction between a single technician and a specific IT issue to form a unique structural relationship between the technician and the statistically important technical issues surfaced in the previous step.

By aggregating these interactions, we calculate the definitive edge weights that represent the strength and frequency of each relationship. The resulting output clearly surfaces the organization's heaviest operational pathways revealing that technicians like Mulan Fa and Andy Davis serve as the primary knowledge brokers and specialists for handling "adobe creative" requests while Minnie Mouse is focused on local computer installations (computer_local) that may require her to also focus on drive data (datum_drive) backups.

In [4]:
# Import libraries
import pandas as pd
import re

# Define function to extract technicians
def extract_technicians_robust(tech_string):
    if pd.isna(tech_string):
        return []
    tech_string = str(tech_string)
    cleaned_string = re.sub(r"\[|\]|\{|\}|\'|\"", "", tech_string)
    technicians = [tech.strip() for tech in re.split(r'[,|;]', cleaned_string)]
    return [tech for tech in technicians if tech]

# Generate the list individual technicians
df['technician_list'] = df['contributing_technicians'].apply(extract_technicians_robust)

# Explode the technical target matrix
edge_df = df.explode('bipartite_targets')

# Explode the source technician matrix
edge_df = edge_df.explode('technician_list')

# Drop null nodes
edge_df = edge_df.dropna(subset=['technician_list', 'bipartite_targets'])

# Isolate coordinates to form the bipartite architecture
edge_list = edge_df[['technician_list', 'bipartite_targets']].copy()

# Rename columns
edge_list.columns = ['Source', 'Target']

# Aggregate identical source-to-target pathways
weighted_edges = edge_list.groupby(['Source', 'Target']).size().reset_index(name='Weight')

# Sort the topological matrix
weighted_edges = weighted_edges.sort_values(by='Weight', ascending=False).reset_index(drop=True)
display(weighted_edges.head(15))

,Source,Target,Weight
0,Mulan Fa,adobe_creative,594
1,Andy Davis,adobe_creative,575
2,Mulan Fa,support_session,498
3,Minnie Mouse,datum_drive,484
4,Minnie Mouse,computer_local,459
5,Andy Davis,serial_number,454
6,Mulan Fa,password_service,420
7,Minnie Mouse,delivery_time,413
8,Andy Davis,datum_drive,409
9,Huey Duck,adobe_creative,406


## Topological Pruning and Bipartite Network Construction

We refine our bipartite network by applying filters to isolate the organization's core operational infrastructure. We implement an edge-weight threshold to eliminate anomalous, one-off interactions so we can  map established and repeatable support relationships.

We apply a k-core decomposition to strip away peripheral dangling nodes that possess only a single connection to the broader network. Finally, the pruned model is serialized into a standardized format, allowing us to export the finalized topology into the Gephi supported file format.



In [5]:
# Import libraries
import networkx as nx
from google.colab import files

# Define Edge Weight Threshold
minimum_edge_weight = 3
core_edges = weighted_edges[weighted_edges['Weight'] >= minimum_edge_weight]

# Initialize the undirected bipartite
G_bipartite = nx.Graph()

# Populate Vertices and Edges with Categorical Attributes
for index, row in core_edges.iterrows():
    source_node = row['Source']
    target_node = row['Target']

    # Inject nodes with explicit class labels for Gephi partitioning
    G_bipartite.add_node(source_node, Node_Class='Technician', label=source_node)
    G_bipartite.add_node(target_node, Node_Class='Technical_Issue', label=target_node)
    G_bipartite.add_edge(source_node, target_node, weight=row['Weight'])

#  Apply Core Decomposition
G_bipartite = nx.k_core(G_bipartite, k=2)

# Serialize and Export the Bipartite Topology
file_name = "core_service_desk_topology.gexf"
nx.write_gexf(G_bipartite, file_name)

# Local Download
print(f"Bipartite network serialized. Initiating download for {file_name}...")
files.download(file_name)

Bipartite network serialized. Initiating download for core_service_desk_topology.gexf...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Core Service Desk Topology

The visualization reveals a highly dense, centralized core component, indicating that the vast majority of IT service desk operations revolve around a shared pool of common technical issues and deeply interconnected personnel.

Conversely, several distinct satellite clusters exist on the structural periphery, representing specialized technical domains handled by isolated subgroups of technicians.

This topology suggests a potential operational vulnerability; while general knowledge is broadly distributed within the main core, the peripheral clusters may indicate siloed expertise and potential single points of failure if those specific technicians become unavailable.

(green = technician, red = technical issue)

In [6]:
from IPython.display import Image, display

# Define the URL
img_url = "https://raw.githubusercontent.com/johnnydrodriguez/data620/main/core_service_desk_topology.png"

# Display the image
display(Image(url=img_url))

## Mapping Technical Dependencies

We project our bipartite data into a one-mode network to exclusively analyze the relationships between different IT systems and software. We pair distinct technical concepts that are documented together within the same service ticket to establish the relationship between them.

Aggregating these pairings reveals how frequently certain system failures or requests co-occur, signaling the technical dependencies within the broader technical ecosystem.

We then apply a threshold and core decomposition to filter out rare anomalies. The resulting model enables us recognize which underlying technologies are fundamentally intertwined in daily operations.

In [7]:
# Import libraries
import pandas as pd
import networkx as nx
from itertools import combinations
from google.colab import files

# Isolate records
multi_issue_records = df['bipartite_targets'].dropna()

# Extract issue-to-issue edges
issue_pair_list = []
for issue_list in multi_issue_records:
    distinct_issues = list(set(issue_list))
    if len(distinct_issues) > 1:
        sorted_issues = sorted(distinct_issues)
        issue_pairs = list(combinations(sorted_issues, 2))
        issue_pair_list.extend(issue_pairs)

# Aggregate identical issue pairings
issue_collab_df = pd.DataFrame(issue_pair_list, columns=['Source', 'Target'])
issue_weights = issue_collab_df.groupby(['Source', 'Target']).size().reset_index(name='Weight')

# Define minimum co-occurrence threshold
min_issue_weight = 2
core_issue_edges = issue_weights[issue_weights['Weight'] >= min_issue_weight]

# Initialize the undirected graph
G_issues = nx.from_pandas_edgelist(
    core_issue_edges,
    source='Source',
    target='Target',
    edge_attr='Weight'
)

# Apply core decomposition
G_issues = nx.k_core(G_issues, k=2)

# Serialize the complete network topology for Gephi
file_name = "core_technical_issues.gexf"
nx.write_gexf(G_issues, file_name)

# Local file download
files.download(file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Core Technical Issues

The visualization of the core technical issues network displays a highly dense, centralized topology, indicating broad structural dependencies within the IT infrastructure.

A large central cluster is dominated by identity and access management concepts, such as "password reset," "account lockout," and "active directory".  These are  intertwined with core enterprise software like "adobe creative" and "duo push."

This clustering reveals that the majority of system failures are not isolated incidents, but rather cascaded events where an issue in one foundational system, such as authentication, triggers a multitude of distinct, downstream ticket categories.

In [8]:
from IPython.display import Image, display

# Define the URL
img_url = "https://raw.githubusercontent.com/johnnydrodriguez/data620/main/core_technical_issues.png"

# Display the image
display(Image(url=img_url))

## Technical Infrastructure Centrality Measures

We quantify the structural importance of each IT concept within the broader infrastructure by calculating degree, betweenness, and eigenvector centrality to identify which systems act as critical operational bottlenecks or have disproportionate systemic influence.

The resulting centrality matrix reveals that "adobe creative" serves as the primary hub of the technical topology; it has both the highest systemic influence and the highest bridging potential within the dataset.

Interestingly, concepts like "datum drive" and "computer local" share similar eigenvector scores, indicating deep integration into standard hardware deployment workflows,  their near-zero betweenness reveals they do not act as informational bridges to other distinct systems.

Conversely, authentication categories such as "password service" and "self service" exhibit significant betweenness, confirming their structural roles as critical operational bottlenecks that can cascade across diverse software and hardware requests.  This is largely due to to the integrated nature of the authentication platform (single sign-on); disruption in these services often affected multiple systems and created cascading effects.

In [9]:
# Import libraries
import pandas as pd
import networkx as nx

# Calculate raw degree centrality
degree_centrality = dict(G_issues.degree())

# Invert edge weights
for u, v, d in G_issues.edges(data=True):
    d['distance'] = 1.0 / d['Weight']

# Calculate betweenness
betweenness = nx.betweenness_centrality(G_issues, weight='distance', normalized=True)

# Calculate eigenvector centrality
eigenvector = nx.eigenvector_centrality(G_issues, weight='Weight', max_iter=1000)

# Compile calculations
issue_centrality_df = pd.DataFrame({
    'Technical_Issue': list(G_issues.nodes()),
    'Degree': [degree_centrality.get(n, 0) for n in G_issues.nodes()],
    'Betweenness': [betweenness.get(n, 0) for n in G_issues.nodes()],
    'Eigenvector': [eigenvector.get(n, 0) for n in G_issues.nodes()]
})

# Sort the matrix descending by eigenvector
issue_centrality_df = issue_centrality_df.sort_values(by='Eigenvector', ascending=False).reset_index(drop=True)

# Format table
issue_centrality_df['Betweenness'] = issue_centrality_df['Betweenness'].round(4)
issue_centrality_df['Eigenvector'] = issue_centrality_df['Eigenvector'].round(4)

# Display centrality matrix
display(issue_centrality_df.head(20))

,Technical_Issue,Degree,Betweenness,Eigenvector
0,adobe_creative,66,0.4858,0.3155
1,datum_drive,58,0.0864,0.2997
2,computer_local,56,0.0036,0.2995
3,delivery_time,55,0.0256,0.2975
4,pre_software,54,0.0000,0.2967
5,deployment_data,54,0.0000,0.2965
6,setup_user,54,0.0000,0.2944
7,application_list,55,0.0000,0.2936
8,machine_upgrade,54,0.0000,0.2935
9,google_chrome,53,0.0440,0.1745


## Topological Mapping of Technician Collaboration

We project our bipartite dataset into a one-mode network  to map the collaborative relationships between IT service desk technicians. By isolating tickets resolved by multiple techs and aggregating these combinations, we generate structural edges that represent shared operational knowledge and teamwork.

We then apply frequency thresholds and a core decomposition algorithm to filter out rare, coincidental interactions, leaving only the established, recurring partnerships. The resulting serialized model allows us to visualize the informal organizational structure, revealing organic sub-teams, isolated personnel, and key collaborative bridges that highlight information flows across the department.


In [10]:
# Import libraries
import pandas as pd
import networkx as nx
from itertools import combinations
from google.colab import files

# Isolate tickets
collaborative_records = df['technician_list'].dropna()
collaborative_records = collaborative_records[collaborative_records.apply(len) > 1]

# Extract human-to-human edges
pair_list = []
for tech_list in collaborative_records:
    sorted_techs = sorted(tech_list)
    pairs = list(combinations(sorted_techs, 2))
    pair_list.extend(pairs)

# Compute collaboration frequency
collab_df = pd.DataFrame(pair_list, columns=['Source', 'Target'])
collab_weights = collab_df.groupby(['Source', 'Target']).size().reset_index(name='Weight')

# Define the minimum number of shared tickets
minimum_collab_weight = 2
core_collab_edges = collab_weights[collab_weights['Weight'] >= minimum_collab_weight]

# Initialize graph object
G_collab = nx.from_pandas_edgelist(
    core_collab_edges,
    source='Source',
    target='Target',
    edge_attr='Weight'
)

# Apply core decomposition
G_collab = nx.k_core(G_collab, k=2)

# Serialize for Gephi file
file_name = "core_technician_collaboration.gexf"
nx.write_gexf(G_collab, file_name)

# Local download
files.download(file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Core Technical Collaboration

The visual representation of the technician collaboration network illustrates a highly centralized operational structure dominated by a dense inner cluster of full-time staff technicians who handle the vast majority of collaborative troubleshooting. This tightly knit group indicates that a concentrated subset of technicians handle the vast majority of collaborative troubleshooting and knowledge sharing within the department.

Conversely, the numerous peripheral nodes operating in relative isolation consist almost exclusively of student technicians. This disparity is highly characteristic of higher education IT environments where transient student workers typically manage routine, independent tasks and only form connections to the core group when escalating complex issues. This was certainly the case out our instition.

The network identifies Mary Poppins as key staff which interacts and is bridge for the subnetwork of student technicians (Mary Poppins is the Student Worker Manager).

In [11]:
from IPython.display import Image, display

# Define the URL
img_url = "https://raw.githubusercontent.com/johnnydrodriguez/data620/main/core_technical_collaboration.png"

# Display the image
display(Image(url=img_url))

## Technician Centrality Measures

We quantify the overall health of our technician network and the specific structural influence of individual technicians. We calculate degree, betweenness, and eigenvector centralities to identify which specific techncians act as critical information brokers or central hubs of expertise.

The global network metrics confirm a highly integrated and efficient human infrastructure within the IT service desk. A network density of 0.2774 paired with a single distinct component indicates that the core collaborative group is fully unified, with nearly **28%** of all possible personnel combinations actively working together to resolve tickets. This is extraordinarily high and signals that technical solution are not siloed within sub tiers.

The average path length of just 1.85 hops demonstrates rapid knowledge transfer;  any two technicians are typically separated by less than two degrees of connection. Finally, the maximum structural diameter of four hops ensures that the most peripheral team members within this core can access the centralized knowledge base with minimal informational bottlenecks requiring an 3 additional technicians in the loop.

The centrality matrix  ranks each technician to reveal the hierarchy of influence and knowledge distribution within the service desk. At the top of the matrix, "star employees" like Andy Davis and Jiminy Cricket exhibit  high systemic influence and betweenness, indicating they act as the critical information brokers bridging various organizational subgroups. In contrast, highly active staff members like Mulan Fa demonstrate high collaboration and influence but possess a betweenness score of absolute zero, indicating she is a highly specialized technician entrenched within a single domain rather than a cross-team bridge.

Finally, the matrix quantifies a structural divide; the student technician tier resides entirely at the bottom with near-zero centrality across all metrics; this confirms they operate on the periphery and remain largely disconnected from the department's core knowledge flow.

The top 5 technicians can be described as follows given their centrality measures:

- Andy Davis: The central anchor of the network, possesses the highest overall structural influence and high-quality collaborative connections.

- Mulan Fa: A highly active domain specialist who collaborates extensively but does not broker information to outside teams.

- Jiminy Cricket: The ultimate knowledge broker who serves as the most critical information bridge across the entire department.

- Huey Duck: A deeply embedded core contributor who partners heavily within the central staff hub but rarely bridges peripheral members.

- Woody Pride: A well-rounded generalist who maintains a high collaborative volume and serves as a reliable secondary knowledge bridge.

In [12]:
# Import libraries
import networkx as nx
import pandas as pd

# Calculate network density
density = nx.density(G_collab)

# Isolate the largest connected component
connected_components = list(nx.connected_components(G_collab))
largest_cc = max(connected_components, key=len)
LCC = G_collab.subgraph(largest_cc)

# Calculate structural diameter and average path length
diameter = nx.diameter(LCC)
avg_path = nx.average_shortest_path_length(LCC)

# Print network statistics
print(f"Network Density: {density:.4f}")
print(f"LCC Diameter: {diameter} hops")
print(f"Average Path Length: {avg_path:.2f} hops")
print(f"Distinct Components: {len(connected_components)}\n")

# Calculate absolute count technicians
degree_centrality = dict(G_collab.degree())

# Invert structural weights
for u, v, d in G_collab.edges(data=True):
    d['distance'] = 1.0 / d['Weight']

# Calculate betweeness centrality
betweenness = nx.betweenness_centrality(G_collab, weight='distance', normalized=True)

# Calculate eigenvector centrality
eigenvector = nx.eigenvector_centrality(G_collab, weight='Weight', max_iter=1000)

# Compile centrality metrics
centrality_df = pd.DataFrame({
    'Technician': list(G_collab.nodes()),
    'Degree_(Collaborators)': [degree_centrality.get(n, 0) for n in G_collab.nodes()],
    'Betweenness_(Brokerage)': [betweenness.get(n, 0) for n in G_collab.nodes()],
    'Eigenvector_(Influence)': [eigenvector.get(n, 0) for n in G_collab.nodes()]
})

# Sort the matrix
centrality_df = centrality_df.sort_values(by='Eigenvector_(Influence)', ascending=False).reset_index(drop=True)

# Format table
centrality_df['Betweenness_(Brokerage)'] = centrality_df['Betweenness_(Brokerage)'].round(4)
centrality_df['Eigenvector_(Influence)'] = centrality_df['Eigenvector_(Influence)'].round(4)

# Display matrix
display(centrality_df.head(100))

Network Density: 0.2774
LCC Diameter: 4 hops
Average Path Length: 1.85 hops
Distinct Components: 1



,Technician,Degree_(Collaborators),Betweenness_(Brokerage),Eigenvector_(Influence)
0,Andy Davis,35,0.2131,0.4606
1,Mulan Fa,40,0.0000,0.4285
2,Jiminy Cricket,36,0.5259,0.3839
3,Huey Duck,33,0.0370,0.3349
4,Woody Pride,37,0.1761,0.3335
5,Aladdin Hassim,29,0.0370,0.2969
6,Tinker Bell,35,0.4626,0.2536
7,Minnie Mouse,12,0.0000,0.2400
8,Scrooge McDuck,25,0.0000,0.1049
9,Mad Hatter,29,0.0000,0.0608


## Conclusion

NLP processing of IT ticket data was challenging in two key ways.  IT ticket data includes many automated or templatized communications or updates that become part of the record; this was noise that needed significant cleaning interspersed with core critical concepts.  On the other end of the spectrum, IT ticket data can be incomplete and sparse when issues are routine or technicians are busy.  Addressing both required many iteration of the NLP analysis until the core set of technical structure surfaced, *give or take.*

That said, I managed many of the technicians included in the 2021 - 2022 dataset and can directly evaluate the outcomes:

- Andy Davis became the anchor technician due to an usual behavioral quirk; he would often proactively seek out and update tickets that were not assigned to him directly.  He was very supportive of the other technicians and would provide updates or guidance in the tickets without being prompted to do so.

- Mulan Fa was so skilled at collaborating with her team members, she recieved a promotion to supervisor in following year to reward her for accomplishments as well as to position her to become an information broker for the broader IT organization.

- Jiminy Cricket was the associate manager for the Service Desk; his role allowed him to connect to and play a central role in many of the issues that arose and connect across the entire IT organization.

- Huey Duck and Woody Pride were well-regarded senior IT technicians focusing on highly complex issues and escalations; typically, they were the technicians you woul reach out when everything else hasn't worked.